# SC-Unmix: vocals and accompaniment
Select **Runtime → Change runtime type → GPU**, then run each cell in order. Upload the SC-Unmix ZIP (GitHub may name it `sc-unmix-main.zip`. Code and trained weights are included). No local installation is needed.

The network predicts vocals, accompaniment is retrieved from **mixture − vocals**. Both WAVs are stereo at 44.1 kHz. This does not separate drums, bass and guitar individually. Audio is processed in your Colab runtime.

In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, zipfile
from google.colab import files

uploaded = files.upload()  # Select sc-unmix.zip or sc-unmix-main.zip.
if len(uploaded) != 1:
    raise ValueError("Upload one release ZIP")
archive = Path(next(iter(uploaded)))
extract_root = Path(tempfile.mkdtemp(prefix="sc-unmix-", dir="/content"))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (extract_root / member.filename).resolve()
        if not target.is_relative_to(extract_root.resolve()):
            raise ValueError("Unsafe archive path")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlinks are not allowed")
    z.extractall(extract_root)
# GitHub may rename the extracted top-level folder to sc-unmix-main. 
# Try to Locate the package by its required files instead of using the archive's display name.

candidates = []
for checkpoint in extract_root.rglob("vocals_best.pt"):
    candidate = checkpoint.parent.parent
    if (candidate / "sc_unmix/separate.py").is_file() and (candidate / "requirements.txt").is_file():
        candidates.append(candidate)
if len(candidates) != 1:
    raise AssertionError(
        "Expected one SC-Unmix release (sc-unmix or sc-unmix-main); found: "
        + repr([str(path) for path in candidates])
    )
CODE_DIR = candidates[0]
CHECKPOINT = CODE_DIR / "checkpoints/vocals_best.pt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(CODE_DIR))
for name in list(sys.modules):
    if name == "sc_unmix" or name.startswith("sc_unmix."):
        del sys.modules[name]
del uploaded
print("Ready:", CODE_DIR)


In [ ]:
import torch
from sc_unmix.separate import load_model, separate_audio, save_estimates, SAMPLE_RATE
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(CHECKPOINT, device)
print("Device:", device, "| parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
if device == "cpu":
    print("CPU inference works but is slower. Consider enabling a GPU runtime.")


## Choose audio
Upload WAV or FLAC, or download a MUSDB sample through `musdb`, as in the Open-Unmix demo. Use audio you have permission to process. The MUSDB option requires an internet connection and downloads the sample dataset, not the full licensed dataset.

In [ ]:
SOURCE = "Upload" #@param ["Upload", "MUSDB sample"]
import numpy as np
import soundfile as sf

if SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one WAV or FLAC file")
    audio_path = Path(next(iter(uploaded)))
    audio, rate = sf.read(audio_path, dtype="float32", always_2d=True)
    track_name = audio_path.stem
    del uploaded
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "musdb"], check=True)
    import musdb
    mus = musdb.DB(root="/content/musdb-sample", download=True, subsets="test")
    track = mus[0]
    audio, rate, track_name = np.asarray(track.audio, dtype=np.float32), track.rate, track.name
print(track_name, "|", rate, "Hz |", round(len(audio) / rate, 1), "seconds")


## Separate and listen
The default uses 11-second windows and 50% overlap. AMP `auto` uses reduced precision on supported GPUs; `none` uses FP32. Preview playback uses one shared gain for all three signals to avoid clipping; saved float WAVs remain unscaled.

In [ ]:
AMP = "auto" #@param ["auto", "none"]
estimates = separate_audio(model, audio, rate, segment_seconds=11.0, overlap=0.5, batch_size=1, amp=AMP)
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sc-unmix-output-", dir="/content"))
paths = save_estimates(estimates, OUTPUT_DIR)
from IPython.display import Audio, display
preview_samples = min(SAMPLE_RATE * 30, estimates["vocals"].shape[-1])
playback_gain = max(1.0, *(float(x[:, :preview_samples].abs().max()) for x in estimates.values()))
for name, waveform in estimates.items():
    print(name, "(first 30 seconds)")
    display(Audio((waveform[:, :preview_samples] / playback_gain).numpy(), rate=SAMPLE_RATE, normalize=False))
print("Outputs:", OUTPUT_DIR)


In [ ]:
download_path = OUTPUT_DIR / "separated_stems.zip"
with zipfile.ZipFile(download_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in paths.values():
        z.write(path, path.name)
files.download(str(download_path))


### Notes
The accompaniment inherits vocal-estimation errors. Outputs are not independently normalized or clipped, so their sum reconstructs the 44.1 kHz mixture up to floating-point rounding. Float WAVs can contain peaks above 1, lower a common playback gain if needed.

Demo workflow inspired by [Open-Unmix](https://github.com/sigsep/open-unmix-pytorch). Architecture based on [SCNet](https://github.com/starrytong/SCNet). See the release README for training, provenance and limitations.